***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')
path_congestion = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Congestion')


# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'RTIS')
    path_config  = os.path.join(path_code, 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'RTIS')
    path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

## Importing

***

In [ ]:
path_in = os.path.join(path_congestion, 'RTIS', 'Massive Data Downloader')

df_hourly = pd.read_csv(os.path.join(path_in, 'Hourly-SACOG-2023-June-Sept-3pm-7pm.csv'))
df_tmc    = pd.read_csv(os.path.join(path_in, 'TMC_Identification.csv'))


Hourly Traffic Data

In [ ]:
df_hourly.head()

In [ ]:
df_hourly['pct_speed'     ] = df_hourly['speed'] / df_hourly['historical_average_speed']
df_hourly['miles_traveled'] = df_hourly['speed'] * df_hourly['travel_time_minutes'     ]

df_hourly['CONGESTED'] = 'No'
df_hourly.loc[df_hourly['pct_speed'] < 0.6, 'CONGESTED'] = 'Yes'

print('')
print('Hours congested: ')
df_hourly.groupby(['CONGESTED'])['measurement_tstamp'].count()

# Above code requires calculation of "free flow speed" using 8pm to 6am of each day (Darren's SQL scripts)

TMC Data

In [ ]:
df_tmc.head()

In [ ]:
print('')
print('Number of road segments: ')
display(df_tmc.groupby(['county'])['tmc'  ].count())

print('')
print('Total miles of road segments: ')
display(df_tmc.groupby(['county'])['miles'].sum())

print('')
print('Average miles of road segments: ')
display(df_tmc.groupby(['county'])['miles'].mean())

print('')

MAP-21 Widget - PHED

In [ ]:

path_in = os.path.join(path_congestion, 'RTIS', 'MAP-21')
df_phed = pd.read_csv(os.path.join(path_in, 'Annual Hours of PHED Per Capita 3pm-7pm_UZA Sacramento.csv'))


df_phed.head()

In [ ]:
df_phed['Year' ] = df_phed['Month'].apply(re_remove_pre)
df_phed['Month'] = df_phed['Month'].apply(re_remove_post)

df_phed['Year'] = df_phed['Year'].astype('int')
df_phed = df_phed.set_index(['Year', 'Month']).reset_index()
df_phed = df_phed.rename(columns = {'PHED (hours)':'PHED'})
df_phed.head()

In [ ]:
df_phed.groupby(['Year'])['PHED'].sum()